# Bias-Variance Tradeoff & Learning Curves

The **most fundamental concept** in machine learning.

- **Bias**: Error from wrong assumptions (underfitting)
- **Variance**: Error from sensitivity to training data (overfitting)
- **Learning Curves**: Diagnose if you need more data or a better model
- **Validation Curves**: Find the sweet spot for hyperparameters

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, load_digits
from sklearn.model_selection import learning_curve, validation_curve, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

## 1. Bias-Variance Decomposition — Visual

```
Total Error = Bias² + Variance + Irreducible Noise
```

| | Low Bias | High Bias |
|---|---------|----------|
| **Low Variance** | **Ideal** (just right) | Underfitting |
| **High Variance** | Overfitting | Terrible model |

In [ ]:
# Demonstrate bias-variance with polynomial regression
np.random.seed(42)
n = 30
X_demo = np.sort(np.random.uniform(0, 1, n))
y_true_func = np.sin(2 * np.pi * X_demo)
y_demo = y_true_func + np.random.normal(0, 0.3, n)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
X_plot = np.linspace(0, 1, 200)

for ax, degree, title in zip(axes, [1, 4, 15],
                              ['Underfitting (High Bias)', 'Good Fit', 'Overfitting (High Variance)']):
    # Fit polynomial
    coeffs = np.polyfit(X_demo, y_demo, degree)
    y_fit = np.polyval(coeffs, X_plot)
    
    ax.scatter(X_demo, y_demo, c='steelblue', s=30, zorder=5, label='Data')
    ax.plot(X_plot, np.sin(2 * np.pi * X_plot), 'g--', alpha=0.5, label='True function')
    ax.plot(X_plot, y_fit, 'r-', linewidth=2, label=f'Degree {degree}')
    ax.set_title(title, fontsize=13)
    ax.legend(fontsize=9)
    ax.set_ylim(-2, 2)
    ax.grid(True, alpha=0.3)

plt.suptitle('Bias-Variance Tradeoff — Polynomial Regression', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Learning Curves — Do You Need More Data?

In [ ]:
# Generate dataset
X, y = make_classification(n_samples=1000, n_features=20, n_informative=10,
                           n_redundant=5, random_state=42)

def plot_learning_curve(estimator, title, X, y, ax, cv=5):
    train_sizes, train_scores, val_scores = learning_curve(
        estimator, X, y, cv=cv, n_jobs=-1, scoring='accuracy',
        train_sizes=np.linspace(0.1, 1.0, 10)
    )
    
    train_mean = train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    val_mean = val_scores.mean(axis=1)
    val_std = val_scores.std(axis=1)
    
    ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')
    ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1, color='orange')
    ax.plot(train_sizes, train_mean, 'o-', color='blue', label='Training')
    ax.plot(train_sizes, val_mean, 'o-', color='orange', label='Validation')
    
    gap = train_mean[-1] - val_mean[-1]
    ax.set_title(f'{title}\nGap: {gap:.3f}', fontsize=12)
    ax.set_xlabel('Training Size'); ax.set_ylabel('Accuracy')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0.5, 1.05)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
models = [
    (LogisticRegression(max_iter=1000), 'Logistic Reg (High Bias)'),
    (RandomForestClassifier(n_estimators=100, random_state=42), 'Random Forest (Balanced)'),
    (DecisionTreeClassifier(random_state=42), 'Decision Tree (High Variance)')
]
for ax, (model, title) in zip(axes, models):
    plot_learning_curve(model, title, X, y, ax)

plt.suptitle('Learning Curves — Diagnosing Bias vs Variance', fontsize=14)
plt.tight_layout()
plt.show()

### How to Read Learning Curves:

| Pattern | Diagnosis | Solution |
|---------|-----------|----------|
| Both scores low, close together | **High bias** (underfitting) | More features, complex model |
| Training high, validation low (big gap) | **High variance** (overfitting) | More data, regularization, simpler model |
| Both converge to high score | **Good fit** | You're done! |
| Validation still rising at end | **Need more data** | Collect more samples |

## 3. Validation Curves — Find Optimal Hyperparameters

In [ ]:
# Validation curve: max_depth for Decision Tree
param_range = np.arange(1, 25)
train_scores, val_scores = validation_curve(
    DecisionTreeClassifier(random_state=42), X, y,
    param_name='max_depth', param_range=param_range,
    cv=5, scoring='accuracy', n_jobs=-1
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# max_depth
axes[0].plot(param_range, train_scores.mean(axis=1), 'o-', color='blue', label='Training')
axes[0].plot(param_range, val_scores.mean(axis=1), 'o-', color='orange', label='Validation')
axes[0].fill_between(param_range, train_scores.mean(axis=1) - train_scores.std(axis=1),
                     train_scores.mean(axis=1) + train_scores.std(axis=1), alpha=0.1, color='blue')
axes[0].fill_between(param_range, val_scores.mean(axis=1) - val_scores.std(axis=1),
                     val_scores.mean(axis=1) + val_scores.std(axis=1), alpha=0.1, color='orange')
best_depth = param_range[np.argmax(val_scores.mean(axis=1))]
axes[0].axvline(best_depth, color='red', linestyle='--', label=f'Best: {best_depth}')
axes[0].set_xlabel('max_depth'); axes[0].set_ylabel('Accuracy')
axes[0].set_title('Validation Curve — max_depth')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# C for SVM
C_range = np.logspace(-3, 3, 15)
train_s, val_s = validation_curve(
    SVC(kernel='rbf'), X, y, param_name='C', param_range=C_range,
    cv=5, scoring='accuracy', n_jobs=-1
)
axes[1].semilogx(C_range, train_s.mean(axis=1), 'o-', color='blue', label='Training')
axes[1].semilogx(C_range, val_s.mean(axis=1), 'o-', color='orange', label='Validation')
axes[1].fill_between(C_range, val_s.mean(axis=1) - val_s.std(axis=1),
                     val_s.mean(axis=1) + val_s.std(axis=1), alpha=0.1, color='orange')
best_C = C_range[np.argmax(val_s.mean(axis=1))]
axes[1].axvline(best_C, color='red', linestyle='--', label=f'Best C: {best_C:.3f}')
axes[1].set_xlabel('C (regularization)'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Validation Curve — SVM C parameter')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Validation Curves — Finding the Sweet Spot', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Model Complexity Spectrum

In [ ]:
# Compare models from simple to complex
models_complexity = [
    ('Logistic Reg', LogisticRegression(max_iter=1000)),
    ('DT depth=3', DecisionTreeClassifier(max_depth=3, random_state=42)),
    ('DT depth=10', DecisionTreeClassifier(max_depth=10, random_state=42)),
    ('DT depth=None', DecisionTreeClassifier(random_state=42)),
    ('RF n=10', RandomForestClassifier(n_estimators=10, random_state=42)),
    ('RF n=100', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('SVM rbf', SVC(kernel='rbf'))
]

train_accs, val_accs = [], []
for name, model in models_complexity:
    train_sc = cross_val_score(model, X, y, cv=5, scoring='accuracy').mean()
    model.fit(X, y)
    train_acc = model.score(X, y)
    train_accs.append(train_acc)
    val_accs.append(train_sc)

fig, ax = plt.subplots(figsize=(12, 5))
x_pos = range(len(models_complexity))
ax.plot(x_pos, train_accs, 'bo-', label='Train Accuracy', markersize=8)
ax.plot(x_pos, val_accs, 'ro-', label='CV Accuracy', markersize=8)
ax.fill_between(x_pos, train_accs, val_accs, alpha=0.2, color='red', label='Overfit Gap')
ax.set_xticks(x_pos)
ax.set_xticklabels([m[0] for m in models_complexity], rotation=30, ha='right')
ax.set_ylabel('Accuracy')
ax.set_title('Model Complexity Spectrum — Train vs CV Score')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Key Takeaways

| Issue | Symptom | Fix |
|-------|---------|-----|
| High Bias | Low train & val scores | More features, complex model |
| High Variance | High train, low val | More data, regularization, simpler model |
| Right Complexity | Good train, close val | Deploy! |

### Practical Checklist:
1. Plot **learning curves** → Do you need more data?
2. Plot **validation curves** → Tune hyperparameters
3. Check **train vs CV gap** → Diagnose bias/variance
4. Use **cross-validation** (never trust a single split)